# Solana (SOL) — Profit-Maximizing Trading Strategy Research

**Goal.** Engineer features, search exhaustively over feature combinations, rule
parameters, labels, models and hyperparameters, and build trading strategies that
**maximize net (post-cost) profit** — then compare honestly against **buy & hold**.

**How this stays honest (so the "profit" is real, not curve-fit):**

| Trap | Defense (in code) |
|---|---|
| Look-ahead in features | Strictly causal indicators; `tests/test_no_lookahead.py` recomputes on truncated history and asserts equality |
| Same-bar execution | Decide at close of *t*, trade at *t+1* (one-bar lag) — `src/backtest.py` |
| CV leakage on time series | `PurgedWalkForward` with purge + embargo — `src/cv.py` |
| Free trading | Fees + slippage charged on every position change |
| Data snooping | A final **holdout** (most recent 15%) is touched **exactly once**; shuffled/shifted null tests; block-bootstrap CIs |

**Pipeline:** load → EDA → 48 causal features → labels + walk-forward CV →
baselines (B&H + tuned rules) → feature-combination search → 10 ML families ×
2 modes (Optuna, profit objective) → **one-shot holdout** → robustness → verdict.

## 1. Setup & data download

The original Kaggle snippet works unchanged (anonymous download). In this repo the
CSV is already under `data/raw/`, so we fall back to it when offline.

In [ ]:
import sys, os, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# make `src` importable whether run from repo root or notebooks/
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

from src.config import CFG
from src import plots

In [ ]:
# --- Original Kaggle download (works in Colab as-is) ---
# import kagglehub
# path = kagglehub.dataset_download("craigdagama/solana-historical-data")
# print("Path to dataset files:", path)

# Robust loader: use the repo copy if present, else download via kagglehub.
def resolve_csv():
    if CFG.raw_csv.exists():
        return CFG.raw_csv
    import kagglehub, glob
    path = kagglehub.dataset_download("craigdagama/solana-historical-data")
    return Path(sorted(glob.glob(os.path.join(path, "*.csv")))[0])

csv_path = resolve_csv()
print("Using dataset:", csv_path)

## 2. Load, clean & explore

`src/data_io.py` normalizes column aliases, parses dates, removes duplicates,
enforces OHLC consistency and **auto-detects the bar granularity** (so every
annualization/window scales automatically — daily here, but hourly data would
just work).

In [ ]:
from src.data_io import load_prices, summarize, detect_granularity
df = load_prices(csv_path)
info = detect_granularity(df)
print(summarize(df).to_string())
print(f"\nIrregular gaps: {info['n_gaps']}  |  periods/year: {info['periods_per_year']}")
df.tail(3)

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
ax[0].plot(df.index, df["close"]); ax[0].set_yscale("log")
ax[0].set_title("SOL close price (log scale)"); ax[0].set_ylabel("USD")
ax[1].plot(df.index, df["volume"], color="gray"); ax[1].set_ylabel("volume")
ax[1].set_title("Volume"); plt.tight_layout(); plt.show()

bh_total = df["close"].iloc[-1] / df["close"].iloc[0] - 1
print(f"Buy & hold over full history: {bh_total:,.0%}  "
      f"(${df['close'].iloc[0]:.2f} -> ${df['close'].iloc[-1]:.2f})")
print("==> This is the benchmark to beat. It is extremely strong.")

In [ ]:
# Stationarity: model returns, not price levels.
from statsmodels.tsa.stattools import adfuller
ret = df["close"].pct_change().dropna()
p_price = adfuller(df["close"].dropna())[1]
p_ret = adfuller(ret)[1]
print(f"ADF p-value — price level : {p_price:.3f}  (>0.05 => non-stationary, as expected)")
print(f"ADF p-value — daily return: {p_ret:.3e} (<0.05 => stationary; features use returns/ratios)")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].hist(ret, bins=80, color="steelblue"); ax[0].set_title("Daily return distribution")
ax[0].axvline(0, color="k", lw=0.8)
ax[1].plot(ret.index, ret.rolling(30).std()*np.sqrt(365)); ax[1].set_title("30d rolling annualized vol")
plt.tight_layout(); plt.show()

## 3. Feature engineering (48 strictly-causal features)

`src/features.py` hand-rolls every indicator (no TA-Lib dependency) so causality is
**guaranteed and unit-tested**. Trend features are expressed as *ratios to price*
(stationary), never raw moving-average levels. Families: returns/momentum, trend,
oscillators (RSI, MACD, stochastic), volatility (realized vol, ATR, Bollinger,
Parkinson), volume, and cyclical calendar.

In [ ]:
from src.pipeline import load_dataset
data = load_dataset(CFG)            # load + features + labels + CV splits + holdout
X, groups = data["X"], data["groups"]
print(f"Feature matrix: {X.shape[0]} rows x {X.shape[1]} features "
      f"(warmup rows dropped from {len(df)})")
for g, cols in groups.items():
    print(f"  {g:11s}: {len(cols):2d}  ->  {', '.join(cols[:5])}{' ...' if len(cols)>5 else ''}")

## 4. Labels & purged walk-forward cross-validation

**Default label:** sign of the next-bar return (with a small cost-aware neutral band).
Horizons {1,2,3,5} and a triple-barrier option are also available and searched.

**CV:** expanding-window walk-forward; training always precedes the test block, with
**purge + embargo** to stop label/serial-correlation leakage. The most recent 15% is
a **holdout** used once at the very end.

In [ ]:
n = len(X)
fig, ax = plt.subplots(figsize=(11, 3))
for i, (tr, va) in enumerate(data["folds"]):
    ax.barh(i, len(tr), color="steelblue")
    ax.barh(i, len(va), left=tr[-1]+1+CFG.embargo_bars, color="orange")
ax.barh(len(data["folds"]), len(data["hold_idx"]), left=len(data["dev_idx"]), color="firebrick")
ax.set_yticks(list(range(len(data["folds"])+1)),
              [f"fold {i+1}" for i in range(len(data["folds"]))] + ["HOLDOUT"])
ax.set_xlabel("row index")
ax.set_title("Walk-forward layout — blue=train, orange=test (purged+embargo), red=final holdout")
plt.tight_layout(); plt.show()
print(f"{len(data['folds'])} folds | dev rows={len(data['dev_idx'])} | holdout rows={len(data['hold_idx'])}")

## 5. Baselines — buy & hold + exhaustive rule grids

Before any ML we grid-search five classic technical strategies (MA-cross, MACD, RSI,
Bollinger, Donchian breakout) over every parameter combination, scored by **mean
out-of-sample net fold return** on the dev folds. This sets the bar ML must clear.

In [ ]:
from src.pipeline import stage_rules
rules = stage_rules(data, CFG)        # cached by run_pipeline.py
show = ["rule", "params", "mean_return", "std_return", "mean_sharpe", "pct_folds_profitable", "score"]
print("Best rule configs per mode (ranked by dev score = mean_ret - 0.25*std):\n")
for mode in ("long_only", "long_short"):
    print(f"### {mode}")
    print(rules[mode][show].head(5).to_string(index=False), "\n")

## 6. Feature-combination search

`src/selection.py`: correlation-prune (|ρ|>0.95) → **permutation importance** on the
OOS folds → **greedy forward selection by net profit**. Selection only ever sees the
training folds. The resulting named sets (`top10/15/25`, `greedy`, `all_pruned`) are
what the model search picks among.

In [ ]:
from src.pipeline import stage_feature_sets
fsets = stage_feature_sets(data, CFG)
ranking = pd.Series(fsets["ranking"]).sort_values(ascending=False)
print("Candidate feature sets:", {k: len(v) for k, v in fsets["sets"].items()})
print("Greedy (profit-selected) set:", fsets["sets"]["greedy"])
fig = plots.importance_bar(ranking, top=20); plt.show()

## 7. Model search — 10 families × 2 modes, profit-maximizing Optuna

Each study searches model hyperparameters **and** the strategy wrapper (label
horizon, signal threshold / neutral band, position sizing, feature set) jointly,
**optimizing mean OOS net return − 0.25·std** through the full cost-aware backtest.
Rules use exhaustive grids; ML uses Optuna TPE + median pruning.

In [ ]:
from src.pipeline import stage_ml
from src.models import ML_MODELS
ml_results = stage_ml(data, fsets, ML_MODELS, CFG)     # all cached

rows = []
for key, r in ml_results.items():
    if not r.get("best_params"): continue
    name, mode = key.split("__")
    a = r.get("best_attrs") or {}
    rows.append({"model": name, "mode": mode, "dev_score": r["best_score"],
                 "dev_mean_return": a.get("mean_return"), "dev_mean_sharpe": a.get("mean_sharpe"),
                 "folds_profitable": a.get("pct_folds_profitable"),
                 "horizon": r["best_params"].get("horizon"),
                 "feature_set": r["best_params"].get("feature_set"),
                 "sizing": r["best_params"].get("sizing")})
dev_lb = pd.DataFrame(rows).sort_values("dev_score", ascending=False).reset_index(drop=True)
print("Dev-set (cross-validated) leaderboard — ranked by profit-stability score:")
dev_lb

## 8. The moment of truth — one-shot holdout evaluation

Every tuned config is now refit on the **full dev set** and evaluated **once** on the
untouched holdout (most recent 15%), net of costs, beside buy & hold. This is the
honest score: nothing here was optimized against this window.

In [ ]:
from src.pipeline import final_leaderboard
from src.metrics import metrics_table
lb, signals = final_leaderboard(data, ml_results, rules, fsets, CFG, top_rules=3)
bh_row = lb[lb.strategy == "BUY & HOLD"].iloc[0]

cols = ["strategy", "type", "mode", "total_return", "sharpe", "sortino",
        "max_drawdown", "calmar", "win_rate", "exposure", "n_trades"]
print(f"HOLDOUT buy & hold: return={bh_row['total_return']:.1%}  Sharpe={bh_row['sharpe']:.2f}  "
      f"maxDD={bh_row['max_drawdown']:.1%}\n")
print("Holdout leaderboard (ranked by net total return):")
lb[cols].head(15)

In [ ]:
# Equity curves: the 4 best holdout strategies vs buy & hold
from src.backtest import backtest, buy_and_hold

hold_ret = data["ret"].iloc[data["hold_idx"]]

def _bt(key, sig):
    funding = CFG.short_funding_bps_per_bar if key.endswith("long_short") else 0.0
    return backtest(sig, hold_ret, CFG.cost_bps_per_side, funding)

# rank all final signals by holdout terminal equity (net of costs)
ranked_keys = sorted(signals.items(), key=lambda kv: _bt(*kv)["equity"].iloc[-1], reverse=True)

named = {"BUY & HOLD": buy_and_hold(hold_ret, CFG.cost_bps_per_side)}
for key, sig in ranked_keys[:4]:
    named[key.replace("__", " | ")] = _bt(key, sig)
plots.equity_curves(named, "Holdout equity: top strategies vs Buy & Hold (net of costs)")
plt.show()

## 9. Winners per mode & deep-dive

In [ ]:
def best_of(mode):
    sub = lb[(lb["mode"] == mode) & (lb["type"] != "benchmark")]
    return sub.iloc[0] if len(sub) else None

for mode in ("long_only", "long_short"):
    w = best_of(mode)
    if w is None: continue
    beat = "BEATS" if w["total_return"] > bh_row["total_return"] else "TRAILS"
    print(f"[{mode}] winner: {w['strategy']}")
    print(f"    holdout net return {w['total_return']:.1%} vs B&H {bh_row['total_return']:.1%}  -> {beat} on return")
    print(f"    Sharpe {w['sharpe']:.2f} vs {bh_row['sharpe']:.2f} | maxDD {w['max_drawdown']:.1%} vs {bh_row['max_drawdown']:.1%} "
          f"| Calmar {w['calmar']:.2f} vs {bh_row['calmar']:.2f}\n")

# drawdown + rolling sharpe of the single best strategy
best_key = ranked_keys[0][0]
funding = CFG.short_funding_bps_per_bar if best_key.endswith("long_short") else 0.0
best_bt = backtest(ranked_keys[0][1], hold_ret, CFG.cost_bps_per_side, funding)
plots.drawdown(best_bt, f"Drawdown — {best_key.replace('__',' | ')} (holdout)"); plt.show()
plots.rolling_sharpe(best_bt["strat_ret"], window=45, ppy=CFG.periods_per_year); plt.show()

## 10. Robustness & overfitting checks

A strategy that survives these is far likelier to generalize: cost sensitivity,
a circular-shift null (does the edge beat randomly time-shifted versions of the same
signal?), and block-bootstrap confidence intervals on the holdout returns.

In [ ]:
from src.robustness import cost_sweep, circular_shift_null, block_bootstrap_ci
best_sig = ranked_keys[0][1].reindex(hold_ret.index)

# 1) cost sensitivity
sweep = {best_key.split("__")[0]: cost_sweep(best_sig, hold_ret, CFG.periods_per_year,
                                             short_funding_bps=funding)}
plots.cost_sweep_plot(sweep); plt.show()
print(sweep[best_key.split("__")[0]].to_string())

In [ ]:
# 2) circular-shift null test
null = circular_shift_null(best_sig, hold_ret, CFG.cost_bps_per_side, n=500,
                           short_funding_bps=funding)
plots.null_distribution(null["nulls"], null["real_return"],
    f"Null test — real vs time-shifted signals (p={null['p_value']:.3f})"); plt.show()
print(f"real holdout return {null['real_return']:.1%} | null mean {null['null_mean']:.1%} "
      f"| null p95 {null['null_p95']:.1%} | p-value {null['p_value']:.3f}")
print("p<0.05 => edge is unlikely to be luck from the signal's trade pattern alone.")

In [ ]:
# 3) block-bootstrap CI on holdout returns
ci = block_bootstrap_ci(best_bt["strat_ret"], block=15, n=2000, ppy=CFG.periods_per_year)
print(f"Holdout total-return 90% CI : [{ci['total_return_ci'][0]:.1%}, {ci['total_return_ci'][1]:.1%}]")
print(f"Holdout Sharpe       90% CI : [{ci['sharpe_ci'][0]:.2f}, {ci['sharpe_ci'][1]:.2f}]")
print(f"P(holdout return < 0)       : {ci['p_return_negative']:.1%}")

## 11. Conclusion

The cell below prints the data-driven verdict from *this* run (re-executing reproduces
it exactly). Read it together with the honesty caveats:

- **Buy & hold of SOL is a very hard benchmark.** Beating it on *total return* is rare;
  the realistic win is **better risk-adjusted return** (Sharpe / Calmar / smaller
  drawdown), which a timing strategy can deliver by sitting out crashes.
- The reported holdout numbers were produced by configs frozen **before** the holdout
  was touched — the defensible, non-overfit estimate of forward performance.
- Net of realistic fees + slippage; the cost-sweep shows how much edge survives higher costs.
- ~1.9k daily bars is a small sample: prefer the simpler, more stable configs and
  treat single-run outperformance with skepticism (hence the null test + bootstrap CIs).

In [ ]:
print("="*72)
print("FINAL VERDICT (holdout, net of costs)")
print("="*72)
print(f"Buy & Hold      : return {bh_row['total_return']:>8.1%} | Sharpe {bh_row['sharpe']:>5.2f} "
      f"| maxDD {bh_row['max_drawdown']:>7.1%}")
for mode in ("long_only", "long_short"):
    w = best_of(mode)
    if w is None: continue
    print(f"Best {mode:<10}: return {w['total_return']:>8.1%} | Sharpe {w['sharpe']:>5.2f} "
          f"| maxDD {w['max_drawdown']:>7.1%} | Calmar {w['calmar']:>5.2f}  <- {w['strategy']}")
ov = lb.iloc[0]
print("-"*72)
print(f"Highest holdout return overall: {ov['strategy']} [{ov['mode']}] = {ov['total_return']:.1%} "
      f"(Sharpe {ov['sharpe']:.2f})")
best_sharpe = lb[lb.type!='benchmark'].sort_values('sharpe', ascending=False).iloc[0]
print(f"Best risk-adjusted (Sharpe)   : {best_sharpe['strategy']} [{best_sharpe['mode']}] "
      f"= Sharpe {best_sharpe['sharpe']:.2f} (return {best_sharpe['total_return']:.1%})")
print("="*72)